# Prize-Sizing: is there a cooling-energy prize to win?

**Question this notebook answers (and *only* this):** how much controllable cooling energy is recoverable on the ExaDigiT FMU, against a realistic baseline, on the real load trace — *before* investing further in workload synthesis / RL / GNN. Method-agnostic: we bound the prize with a baseline and a strong reference controller; we do NOT test a learned method here.

**Objective** (single source of truth in `rollout.POWER_VARS`):
`P_cooling = W_flow_CT + W_flow_CTWP + W_flow_HTWP + Σ_k W_flow_CDUP`  (watts; η=0.85 baked in → input power). `E = Σ P·Δt`.

**Decision rule** (fractional `ΔE = (E_base − E_ref)/E_base`):
- `ΔE ≈ 1–3%`  → baseline already near-optimal; thesis in trouble.
- `ΔE ≈ 5–15%` → real & publishable; synthesis/RL justified.
- `ΔE > 20%`  → suspect a mis-defined objective (e.g. pump term) before celebrating.

**Assumptions to verify before trusting numbers** (see code comments):
1. `POWER_VARS` names match the FMU (Step 0 checks this).
2. `T_MAX_K` — the cabinet temperature limit — is a **placeholder**; set it to the real spec.
3. `policies.BASELINE_CDU` — the baseline CDU setpoints — is a **placeholder** (range midpoints); set to true current practice.

In [ ]:
# Run this notebook from the validation/ directory so `import rollout` resolves.
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import rollout, policies
importlib.reload(rollout); importlib.reload(policies)

# ---- experiment constants ----
STEP_SIZE    = 15.0          # s  (FMU ZOH step)
STOP_TIME    = 24 * 60 * 60  # s  (one day = one pass over the trace)
EXOGEN_GEN_V = 1            # which exogenous pipeline (1 or 2) — RECORD THIS in the writeup

# !! ASSUMPTION TO VERIFY !!  cabinet temperature limit (performance/safety constraint).
# Placeholder: 45 °C. Replace with the real CDU/cabinet spec before trusting feasibility.
T_MAX_K = 273.15 + 45.0

RUN_KW = dict(stop_time=STOP_TIME, step_size=STEP_SIZE, exogen_gen_v=EXOGEN_GEN_V)
print(f'steps/run = {int(STOP_TIME//STEP_SIZE)},  T_MAX_K = {T_MAX_K:.2f}')

## Step 0 — verify the objective variable names
If `compute_P_cooling` raises a KeyError, the printed `W_flow` list shows the real names — correct `rollout.POWER_VARS` accordingly.

In [ ]:
_probe = rollout.make_env(**RUN_KW)
_probe.reset()
print('FMU W_flow variables found:')
for n in rollout.list_power_vars(_probe):
    print('  ', n)
print('\nP_cooling at t=0:', rollout.compute_P_cooling(_probe.fmu), 'W')
del _probe

## Step 1 — baseline (the bar to beat)
RL deltas = 0 → CT tracks `wetbulb + 10°F` rule, CDU at nominal. This is current-practice, a stronger bar than ASHRAE.

In [ ]:
df_base = rollout.run_policy(policies.baseline_policy, name='baseline', **RUN_KW)
s_base  = rollout.summarize(df_base, T_MAX_K, STEP_SIZE)
E_base  = s_base['E_cooling_J']
s_base

In [ ]:
# Free early read: thermal margin + cooling power over the day.
# Large margin at baseline -> expect a real prize; temps riding the limit -> expect little.
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(df_base['t_s'] / 3600, df_base['P_cooling_W'] / 1e3)
ax[0].set_ylabel('P_cooling [kW]'); ax[0].set_title('Baseline cooling power')
ax[1].plot(df_base['t_s'] / 3600, df_base['T_cab_max_K'] - 273.15, label='max cabinet T')
ax[1].axhline(T_MAX_K - 273.15, color='r', ls='--', label='T_max')
ax[1].set_ylabel('T [°C]'); ax[1].set_xlabel('hour'); ax[1].legend()
plt.tight_layout(); plt.show()
print(f"baseline E = {s_base['E_cooling_kWh']:.1f} kWh | feasible={s_base['feasible']} | margin={s_base['T_cab_margin_K']:.1f} K")

## Step 2 — best static setpoints (ΔE floor)
Grid over the high-leverage knobs (CDU supply temp, pump dp, CT approach), held constant. The gap baseline→best-static is a **floor** on the prize. Coarsen the grid if it's slow (each grid point is one full rollout).

In [ ]:
tsec_grid = np.linspace(-1, 1, 5)   # CDU supply temp knob
dp_grid   = np.linspace(-1, 1, 5)   # pump dp knob
ct_grid   = [2, 4, 6]               # CT approach: colder / rule / warmer

records = []
for ta in tsec_grid:
    for da in dp_grid:
        for ct in ct_grid:
            pol = policies.make_constant_policy(policies.make_cdu_vec(ta, da), ct_action=ct)
            df  = rollout.run_policy(pol, name='static', **RUN_KW)
            s   = rollout.summarize(df, T_MAX_K, STEP_SIZE)
            s.update(tsec_a=ta, dp_a=da, ct=ct)
            records.append(s)
sweep = pd.DataFrame(records)
feasible_sweep = sweep[sweep['feasible']].sort_values('E_cooling_J')
print(f'{len(feasible_sweep)}/{len(sweep)} grid points feasible')
feasible_sweep.head(10)

In [ ]:
E_static = feasible_sweep['E_cooling_J'].min()
dE_floor = (E_base - E_static) / E_base
print(f'ΔE_floor (best static vs baseline) = {dE_floor*100:.2f}%')

## Step 3 — oracle (ΔE ceiling) — OPTIONAL / SLOW
Time-varying clairvoyant schedule = true upper bound. Each candidate is a full rollout, so start with few segments / low maxiter and parallelize later. Build this only if the floor looks promising.

In [ ]:
RUN_ORACLE = False  # set True when ready (expect minutes–hours depending on settings)
E_oracle = None
if RUN_ORACLE:
    best_params, df_oracle, res = policies.optimize_oracle(
        rollout.run_policy, rollout.summarize,
        n_segments=6, T_max_K=T_MAX_K, step_size=STEP_SIZE,
        stop_time=STOP_TIME, exogen_gen_v=EXOGEN_GEN_V, maxiter=10)
    s_oracle = rollout.summarize(df_oracle, T_MAX_K, STEP_SIZE)
    E_oracle = s_oracle['E_cooling_J']
    print(s_oracle)

## Step 4 — verdict

In [ ]:
def verdict(dE):
    if dE < 0.03:  return 'TROUBLE: baseline near-optimal, thesis at risk'
    if dE < 0.05:  return 'MARGINAL: borderline, check robustness'
    if dE <= 0.20: return 'PRIZE EXISTS: real & publishable, synthesis/RL justified'
    return 'SUSPECT: >20%, check objective (pump term / T_max / baseline) before celebrating'

print(f'E_baseline      = {E_base/3.6e6:.1f} kWh')
print(f'ΔE_floor (static)  = {dE_floor*100:5.2f}%  -> {verdict(dE_floor)}')
if E_oracle is not None:
    dE_ceiling = (E_base - E_oracle) / E_base
    print(f'ΔE_ceiling (oracle) = {dE_ceiling*100:5.2f}%  -> {verdict(dE_ceiling)}')
else:
    print('ΔE_ceiling (oracle) = not run (RUN_ORACLE=False)')

print('\nReminders: ΔE is the FMU-measured (fractional) prize; η cancels so it is unbiased.')
print('Real-world transfer has two opposing out-of-model effects (part-load penalty vs')
print('efficiency-curve opportunity) — do not adjust the threshold for them here.')